Setup and Load

In [1]:
import pandas as pd, numpy as np, re, sys
from pathlib import Path
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
df = pd.read_excel(ROOT / "data/raw/Challenge_Data.xls")
print(df.shape, df.dtypes, sep="\n")

(5899, 8)
Col1                              str
Col2                           object
Col3                          float64
Col4                              str
Col5                   datetime64[us]
Col6                              str
Col7                              str
ClassificationLabel               str
dtype: object


Inspecting Sample Values in Col1, Col2, Col4, Col6 (Highest Cardinality in these columns)

In [2]:
for c in ["Col1", "Col2", "Col4", "Col6"]:
    print(f"\n===== {c} | nunique={df[c].nunique()} | nulls={df[c].isna().sum()}")
    print(df[c].dropna().sample(30, random_state=42).to_list())


===== Col1 | nunique=313 | nulls=0
['Word16 Word17 Word18 Word7', 'Word30 Word31', 'Word16 Word17 Word18 Word7', 'Word162 Word7', 'Word40 Word41 Word4 Word5', 'Word40 Word41 Word4 Word5', 'Word16 Word17 Word18 Word7', 'Word38 Word4 Word39', 'Word257 Word258 Word259 Word5', 'Word30 Word31', 'Word160 Word161 Word61 Word45 Word39', 'Word16 Word17 Word18 Word7', 'Word8 Word9 Word10 Word11 Word5', 'Word240 Word276 Word3 Word4 Word5', 'Word8 Word9 Word10 Word11 Word5', 'Word153 Word154 Word109 Word155 Word31', 'Word160 Word161 Word61 Word45 Word39', 'Word16 Word17 Word18 Word7', 'Word122 Word123 Word21 Word124 Word5', 'Word16 Word17 Word18 Word7', 'Word116 Word117 Word118 Word119 Word7', 'Word110 Word22 Word4 Word5', 'Word16 Word17 Word18 Word7', 'Word179 Word180 Word181 Word182', 'Word96 Word3 Word4 Word5', 'Word8 Word9 Word10 Word11 Word5', 'Word16 Word17 Word18 Word7', 'Word33 Word34 Word35 Word7', 'Word16 Word17 Word18 Word7', 'Word30 Word31']

===== Col2 | nunique=4818 | nulls=0
['RY13

The remaining columns: amount, date, document status, and Col2's type mix

In [3]:
print("=== Col3 (numeric)")
print(df["Col3"].describe())
print("zeros:", (df["Col3"] == 0).sum(), "| negatives:", (df["Col3"] < 0).sum(),
      "| nulls:", df["Col3"].isna().sum())
print("top repeated values:\n", df["Col3"].value_counts().head(10))
print(df["Col3"].sample(20, random_state=42).to_list())

print("\n=== Col5 (datetime)")
print(df["Col5"].min(), "to", df["Col5"].max(), "| nulls:", df["Col5"].isna().sum())
print("distinct times of day:", df["Col5"].dt.time.nunique())
print(df["Col5"].sample(10, random_state=42).to_list())

print("\n=== Col7")
print(df["Col7"].value_counts(dropna=False))

print("\n=== Col2 type mix")
print(df["Col2"].map(type).value_counts())

=== Col3 (numeric)
count    5.899000e+03
mean     1.358787e+05
std      9.301641e+05
min     -2.691116e+05
25%      3.600000e+02
50%      1.093200e+03
75%      4.520040e+03
max      8.799524e+06
Name: Col3, dtype: float64
zeros: 2 | negatives: 69 | nulls: 0
top repeated values:
 Col3
2520.00       82
1260.00       75
360.00        73
48.00         64
8799523.59    59
180.00        57
140.40        54
1080.00       51
240.00        45
315.00        43
Name: count, dtype: int64
[2520.0, 728.0, 6300.0, 10500.0, 2570.0, 4150.0, 3780.0, 3862.5, 27300.0, 467.5, 50.0, 540.0, 24975.0, 950.0, 55.9, 9574.15, 55.0, 1123.2, 10800.0, 1394.4]

=== Col5 (datetime)
2016-10-01 00:00:00 to 2018-12-31 00:00:00 | nulls: 0
distinct times of day: 1
[Timestamp('2018-09-30 00:00:00'), Timestamp('2018-05-31 00:00:00'), Timestamp('2018-05-31 00:00:00'), Timestamp('2018-03-01 00:00:00'), Timestamp('2018-07-31 00:00:00'), Timestamp('2018-09-30 00:00:00'), Timestamp('2018-06-30 00:00:00'), Timestamp('2018-11-30 00

Normalize Labels: 11 strings for 6 classes

In [4]:
def normalize_label(s):
    d = re.search(r"\d+", str(s))
    return f"Category_{d.group()}" if d else np.nan

raw = df["ClassificationLabel"].value_counts(dropna=False)
df["label"] = df["ClassificationLabel"].map(normalize_label)
clean = df["label"].value_counts(dropna=False)
print(raw, "\n\n", clean)

assert df["label"].nunique() == 6, "expected 6 classes"
assert clean.sum() == 5899, "row count changed"
assert df["label"].isna().sum() == 0, "unparsed label"
print("passed")

ClassificationLabel
Category_1     5204
Category_2      627
Category4        16
Category_3       12
category_1       11
Categry_6        10
Category 3       10
Category2         4
Category_6        2
Category 5        2
Category _3       1
Name: count, dtype: int64 

 label
Category_1    5215
Category_2     631
Category_3      23
Category_4      16
Category_6      12
Category_5       2
Name: count, dtype: int64
passed


Amount Anomalies Checks: The maximum repeats 59 times


In [5]:
mx = df["Col3"].max()
print("rows at max:", (df["Col3"] == mx).sum())
print(df.loc[df["Col3"] == mx, "label"].value_counts())
print(df.loc[df["Col3"] == mx, ["Col1", "Col6", "Col7", "Col5"]].head())

print("\nnegatives by label:\n", df.loc[df["Col3"] < 0, "label"].value_counts())

print("\ndistinct dates:", df["Col5"].nunique())
print("month-end share:", (df["Col5"] == df["Col5"] + pd.offsets.MonthEnd(0)).mean().round(3))
print(df["Col5"].dt.to_period("M").value_counts().sort_index().head(12))

print("\nfloat Col2 rows:")
print(df.loc[df["Col2"].map(lambda x: isinstance(x, float)), ["Col2", "Col1", "label"]])

rows at max: 59
label
Category_1    59
Name: count, dtype: int64
                     Col1                             Col6   Col7       Col5
26   Word44 Word45 Word46                  Word563 Word587  NoDoc 2018-12-31
171  Word44 Word45 Word46                  Word563 Word250  NoDoc 2018-12-31
186  Word44 Word45 Word46     Word563 Word366 Word56 Word4  NoDoc 2018-12-31
259  Word44 Word45 Word46  Word185 Word630 Word450 Word631  NoDoc 2018-12-31
387  Word44 Word45 Word46  Word560 Word561 Word575 Word598   Doc1 2018-12-31

negatives by label:
 label
Category_1    67
Category_5     2
Name: count, dtype: int64

distinct dates: 17
month-end share: 0.68
Col5
2016-10     36
2017-04      2
2017-10      1
2017-12    229
2018-01    312
2018-02    533
2018-03    502
2018-04    528
2018-05    526
2018-06    535
2018-07    496
2018-08    580
Freq: M, Name: count, dtype: int64

float Col2 rows:
          Col2                        Col1       label
2464  108542.1  Word90 Word91 Word92 Word7  Catego

Cleaning the text columns: does trimming or casing change anything? - No

In [6]:
phrase_cols = ["Col1", "Col4", "Col6"]
id_col = "Col2"
flat_cols = ["Col7"]
all_cats = phrase_cols + [id_col] + flat_cols

def clean_text(s):
    if pd.isna(s):
        return np.nan
    return re.sub(r"\s+", " ", str(s)).strip()

def clean_id(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, float) and x.is_integer():
        x = int(x)
    return str(x).strip()

# capture original storage type before casting destroys it
df["col2_raw_type"] = df[id_col].map(
    lambda x: "float" if isinstance(x, float) else ("int" if isinstance(x, int) else "str"))

before = {c: df[c].nunique() for c in all_cats}
df[id_col] = df[id_col].map(clean_id)
for c in phrase_cols + flat_cols:
    df[c] = df[c].map(clean_text)

trimmed = {c: df[c].nunique() for c in all_cats}
lowered = {c: df[c].str.lower().nunique() for c in all_cats}
pd.DataFrame({"raw": before, "trimmed": trimmed, "lowered": lowered})

,raw,trimmed,lowered
Col1,313,313,313
Col4,2032,2032,2032
Col6,196,196,196
Col2,4818,4818,4818
Col7,4,4,4


Col2 checking whether the int/text split is real signal or an Excel artifact

In [7]:
c2 = df["Col2"].astype(str)
df["col2_is_numeric"] = c2.str.fullmatch(r"\d+")

# prove the regex reproduces the dtype split exactly
print(pd.crosstab(df["col2_raw_type"], df["col2_is_numeric"]))
print("mismatch rows:", ((df["col2_raw_type"] == "int") != df["col2_is_numeric"]).sum())

print("\nclass share by col2_is_numeric:")
print(pd.crosstab(df["col2_is_numeric"], df["label"], normalize="index").mul(100).round(2))
print(df["col2_is_numeric"].value_counts())

print("\nCol7 by col2_is_numeric:")
print(pd.crosstab(df["col2_is_numeric"], df["Col7"]))

col2_is_numeric  False  True 
col2_raw_type                
float                3      0
int                  0   3526
str               2370      0
mismatch rows: 0

class share by col2_is_numeric:
label            Category_1  Category_2  Category_3  Category_4  Category_5  \
col2_is_numeric                                                               
False                 96.88        1.85        0.84        0.17        0.08   
True                  82.70       16.65        0.09        0.34        0.00   

label            Category_6  
col2_is_numeric              
False                  0.17  
True                   0.23  
col2_is_numeric
True     3526
False    2373
Name: count, dtype: int64

Col7 by col2_is_numeric:
Col7             Doc1  Doc2  Doc3  NoDoc
col2_is_numeric                         
False             403     9    30   1931
True              705    26    91   2704


Col2: nesting, frequency, and inflated mutual information

In [8]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

c2 = df["Col2"].astype(str)

nest = df.groupby(c2)["Col1"].nunique()
print("Col2 groups with >1 Col1:", (nest > 1).sum(), "of", len(nest))

vc = c2.value_counts()
print("\nappear once:", (vc == 1).sum(), f"({(vc == 1).sum() / len(vc):.1%} of values)")
print("rows covered by singletons:", (vc == 1).sum(), f"({(vc == 1).sum() / len(df):.1%} of rows)")
print(vc.head(10))

y = LabelEncoder().fit_transform(df["label"])
mi_raw = mutual_info_classif(
    LabelEncoder().fit_transform(c2).reshape(-1, 1), y,
    discrete_features=True, random_state=42)[0]
print("\nMI raw Col2:", round(mi_raw, 4))

Col2 groups with >1 Col1: 39 of 4818

appear once: 4247 (88.1% of values)
rows covered by singletons: 4247 (72.0% of rows)
Col2
201811-00046    59
201806-00045    35
4.80P+11        22
4.80W+11        19
4.80J+11        19
4.80F+11        17
4.80U+11        17
4.80Q+11        14
4.80I+11        14
4.80R+11        14
Name: count, dtype: int64

MI raw Col2: 0.398


Figure setup and How far frequency grouping can go

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGDIR = ROOT / "artifacts" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGDIR / f"{name}.png", dpi=120)
    plt.close(fig)
    print("saved", name)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rows = []
for ax, c in zip(axes, ["Col1", "Col4", "Col6"]):
    vc = df[c].value_counts()
    cum = vc.cumsum() / len(df)
    ax.plot(range(1, len(cum) + 1), cum.values)
    ax.axhline(0.95, ls="--", c="r", lw=1)
    ax.set_title(c); ax.set_xlabel("distinct values (ranked)"); ax.set_ylabel("cumulative row share")
    rows.append({
        "col": c, "n_unique": len(vc),
        "n_for_50%": int((cum < 0.50).sum() + 1),
        "n_for_80%": int((cum < 0.80).sum() + 1),
        "n_for_95%": int((cum < 0.95).sum() + 1),
        "n_for_99%": int((cum < 0.99).sum() + 1),
        "singletons": int((vc == 1).sum()),
    })
save(fig, "07_cumulative_coverage")
pd.DataFrame(rows).set_index("col")

saved 07_cumulative_coverage


,n_unique,n_for_50%,n_for_80%,n_for_95%,n_for_99%,singletons
col,,,,,,
Col1,313,10,44,128,255,108
Col4,2032,147,1006,1891,2033,1335
Col6,196,3,16,68,141,52


Col4 Missingness tested as a feature

In [10]:
col4_null = df["Col4"].isna()
cnt = pd.crosstab(col4_null, df["label"])
pct = pd.crosstab(col4_null, df["label"], normalize="index").mul(100).round(2)
cnt.index = pct.index = ["Col4 present", "Col4 missing"]

summary = pd.concat([cnt, pct], keys=["count", "pct"], axis=1)
summary.to_csv(ROOT / "artifacts" / "col4_nullness_by_class.csv")
summary

count                                              \
label        Category_1 Category_2 Category_3 Category_4 Category_5   
Col4 present       5075        618         23         16          2   
Col4 missing        140         13          0          0          0   

                               pct                                   \
label        Category_6 Category_1 Category_2 Category_3 Category_4   
Col4 present         12      88.32      10.76        0.4       0.28   
Col4 missing          0      91.50       8.50        0.0       0.00   

                                    
label        Category_5 Category_6  
Col4 present       0.03       0.21  
Col4 missing       0.00       0.00

 Col5 checked for drift across the period

In [11]:
df["period"] = df["Col5"].dt.to_period("M")
by_month = pd.crosstab(df["period"], df["label"])
share = pd.crosstab(df["period"], df["label"], normalize="index").round(4) * 100
print(by_month, "\n\n", share)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
by_month.sum(axis=1).plot(kind="bar", ax=axes[0])
axes[0].set_ylabel("rows"); axes[0].set_title("volume by period")
share.plot(kind="bar", stacked=True, ax=axes[1])
axes[1].set_ylabel("% of period"); axes[1].set_title("class share by period")
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
save(fig, "09_class_share_by_month")

label    Category_1  Category_2  Category_3  Category_4  Category_5  \
period                                                                
2016-10          36           0           0           0           0   
2017-04           2           0           0           0           0   
2017-10           1           0           0           0           0   
2017-12         176          51           0           2           0   
2018-01         283          27           0           2           0   
2018-02         484          48           0           1           0   
2018-03         440          61           0           1           0   
2018-04         481          43           0           1           0   
2018-05         474          45           6           0           0   
2018-06         468          61           3           1           0   
2018-07         425          64           3           4           0   
2018-08         498          77           4           0           0   
2018-0


Col3: Distribution, Skew, and Outliers

In [12]:
x = df["Col3"]
print("skew:", round(x.skew(), 2), "| kurtosis:", round(x.kurtosis(), 2))
print("skew of signed log:", round(np.sign(x).mul(np.log1p(x.abs())).skew(), 2))

print("\ntop 20 repeated values:")
print(x.value_counts().head(20))
print("\nround-number share (multiple of 10):", round((x % 10 == 0).mean(), 3))
for s in [0, -1, 1, 9999, 99999, 999999, -9999]:
    n = (x == s).sum()
    if n: print(f"sentinel candidate {s}: {n} rows")

# reversal pairs: same magnitude appearing with both signs
mags = x[x != 0].abs().value_counts()
pairs = [m for m in mags.index if (x == m).any() and (x == -m).any()]
print("\nmagnitudes present with both signs:", len(pairs))
for m in sorted(pairs, key=lambda v: -(x.abs() == v).sum())[:5]:
    print(f"  {m}: +{(x == m).sum()} rows / -{(x == -m).sum()} rows")

# large repeated values vs Col2 document counts
big = x.value_counts()
big = big[(big.values >= 20) & (np.abs(big.index.values) > 100000)]
print("\nlarge repeated values:")
for v, n in big.items():
    docs = df.loc[x == v, "Col2"].nunique()
    print(f"  {v}: {n} rows, {docs} distinct Col2")

df["col3_signed_log"] = np.sign(x) * np.log1p(x.abs())

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].hist(x, bins=100); axes[0, 0].set_title("Col3 raw")
axes[0, 1].hist(df["col3_signed_log"], bins=100); axes[0, 1].set_title("Col3 signed log")
axes[1, 0].boxplot(x, orientation="horizontal"); axes[1, 0].set_title("Col3 raw boxplot")
order = df["label"].value_counts().index.tolist()
axes[1, 1].boxplot(
    [df.loc[df["label"] == c, "col3_signed_log"] for c in order],
    tick_labels=order, orientation="horizontal")
axes[1, 1].set_title("signed log by class")
save(fig, "10_col3_distribution")

skew: 8.39 | kurtosis: 73.21
skew of signed log: -2.05

top 20 repeated values:
Col3
 2520.00       82
 1260.00       75
 360.00        73
 48.00         64
 8799523.59    59
 180.00        57
 140.40        54
 1080.00       51
 240.00        45
 315.00        43
 540.00        42
 576.00        38
 5040.00       36
 720.00        36
 2570126.23    35
-269111.58     35
 2829197.98    35
 270.00        34
 269111.58     34
 900.00        33
Name: count, dtype: int64

round-number share (multiple of 10): 0.358
sentinel candidate 0: 2 rows

magnitudes present with both signs: 6
  269111.58: +34 rows / -35 rows
  48.0: +64 rows / -1 rows
  44473.31: +19 rows / -15 rows
  4100.0: +23 rows / -1 rows
  1155.0: +11 rows / -2 rows

large repeated values:
  8799523.59: 59 rows, 1 distinct Col2
  2570126.23: 35 rows, 34 distinct Col2
  -269111.58: 35 rows, 35 distinct Col2
  2829197.98: 35 rows, 1 distinct Col2
  269111.58: 34 rows, 34 distinct Col2
saved 10_col3_distribution


Mutual information for every feature

In [13]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

y = LabelEncoder().fit_transform(df["label"])
p = df["label"].value_counts(normalize=True)
H = -(p * np.log(p)).sum()
print("label entropy:", round(H, 4), "\n")

feats = {
    "Col1": (df["Col1"].astype(str), True),
    "Col2": (df["Col2"].astype(str), True),
    "Col3": (df["Col3"], False),
    "Col4": (df["Col4"].fillna("__MISSING__").astype(str), True),
    "Col5_month": (df["Col5"].dt.to_period("M").astype(str), True),
    "Col6": (df["Col6"].astype(str), True),
    "Col7": (df["Col7"].astype(str), True),
}

rows = []
for name, (s, is_disc) in feats.items():
    v = LabelEncoder().fit_transform(s) if is_disc else s.values
    mi = mutual_info_classif(v.reshape(-1, 1), y, discrete_features=is_disc, random_state=42)[0]
    rows.append({"feature": name, "n_unique": s.nunique(),
                 "MI": round(mi, 4), "share_of_entropy": round(mi / H, 3)})

mi_df = pd.DataFrame(rows).sort_values("MI", ascending=False).set_index("feature")
mi_df.to_csv(ROOT / "artifacts" / "mutual_information.csv")
mi_df

label entropy: 0.401 



,n_unique,MI,share_of_entropy
feature,,,
Col2,4818,0.3980,0.992
Col4,2033,0.3653,0.911
Col1,313,0.2705,0.674
Col6,196,0.2185,0.545
Col3,2397,0.1080,0.269
Col7,4,0.0158,0.039
Col5_month,16,0.0115,0.029


Cramer's V between categorical pairs, and where it disagrees with mutual information

In [14]:
from scipy.stats import chi2_contingency
from itertools import combinations

cat_cols = ["Col1", "Col2", "Col4", "Col6", "Col7", "label"]

def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    phi2 = chi2 / n
    r, k = ct.shape
    # bias correction, keeps big tables from looking artificially strong
    phi2c = max(0, phi2 - (r - 1) * (k - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2c / max(1e-12, min(rc - 1, kc - 1)))

V = pd.DataFrame(np.eye(len(cat_cols)), index=cat_cols, columns=cat_cols)
for a, b in combinations(cat_cols, 2):
    v = cramers_v(df[a].fillna("__MISSING__").astype(str),
                  df[b].fillna("__MISSING__").astype(str))
    V.loc[a, b] = V.loc[b, a] = round(v, 3)

V.to_csv(ROOT / "artifacts" / "cramers_v.csv")
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(V.values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(cat_cols)), cat_cols, rotation=45)
ax.set_yticks(range(len(cat_cols)), cat_cols)
for i in range(len(cat_cols)):
    for j in range(len(cat_cols)):
        ax.text(j, i, V.values[i, j], ha="center", va="center", color="w", fontsize=8)
fig.colorbar(im)
save(fig, "12_cramers_v")
V

saved 12_cramers_v


,Col1,Col2,Col4,Col6,Col7,label
Col1,1.000,0.426,0.722,0.584,0.667,0.656
Col2,0.426,1.000,0.553,0.303,0.395,0.426
Col4,0.722,0.553,1.000,0.575,0.629,0.780
Col6,0.584,0.303,0.575,1.000,0.961,0.617
Col7,0.667,0.395,0.629,0.961,1.000,0.157
label,0.656,0.426,0.780,0.617,0.157,1.000


Duplicates, Label Conflicts, and the Accuracy Ceiling

In [15]:
feature_cols = ["Col1", "Col2", "Col3", "Col4", "Col5", "Col6", "Col7"]

# duplicates: report only, nothing removed
dup_mask = df.duplicated(subset=feature_cols + ["label"], keep=False)
print("rows in exact-duplicate groups:", dup_mask.sum())
print("rows a dedup would have removed:", df.duplicated(subset=feature_cols + ["label"]).sum())
print("\nlabels of duplicated rows:\n", df.loc[dup_mask, "label"].value_counts())

sizes = df[dup_mask].groupby(feature_cols, dropna=False).size().sort_values(ascending=False)
print("\nduplicate groups:", len(sizes), "| largest sizes:", sizes.head(10).tolist())
# same features, different label
g = df.groupby(feature_cols, dropna=False)["label"].nunique()
conflict = g[g > 1]
print("\nconflicting feature groups:", len(conflict))
if len(conflict):
    idx = df.set_index(feature_cols).index.isin(conflict.index)
    conf = df[idx]
    lost = conf.groupby(feature_cols, dropna=False)["label"].apply(
        lambda s: len(s) - s.value_counts().iloc[0]).sum()
    print("rows involved:", len(conf), "| unavoidable errors:", lost)
    print("accuracy ceiling:", round(1 - lost / len(df), 4))

df["group_key"] = (df[feature_cols].astype(object).fillna("__NA__")
                   .astype(str).agg("|".join, axis=1).factorize()[0])
print("\nrows:", len(df), "| distinct groups:", df["group_key"].nunique())

rows in exact-duplicate groups: 1215
rows a dedup would have removed: 679

labels of duplicated rows:
 label
Category_1    1002
Category_2     213
Name: count, dtype: int64

duplicate groups: 535 | largest sizes: [5, 5, 5, 5, 5, 4, 4, 4, 4, 4]

conflicting feature groups: 1
rows involved: 5 | unavoidable errors: 2
accuracy ceiling: 0.9997

rows: 5899 | distinct groups: 5219


Leakage Check: Values that map to one class every time

In [16]:
from scipy.stats import binomtest

base = df["label"].value_counts(normalize=True)

df["col5_month"] = df["Col5"].dt.to_period("M").astype(str)
check_cols = ["Col1", "Col4", "Col6", "Col7", "col2_is_numeric", "Col3", "col5_month"]
MIN_N = 5

rows = []
for c in check_cols:
    s = df[c].fillna("__MISSING__").astype(str)
    for v, lab in df.assign(_v=s).groupby("_v")["label"]:
        n = len(lab)
        if n < MIN_N:
            continue
        top = lab.value_counts()
        cls, k = top.index[0], top.iloc[0]
        if k == n:
            rows.append({
                "col": c, "value": v[:45], "n": n, "class": cls,
                "p_by_chance": binomtest(k, n, base[cls], alternative="greater").pvalue,
            })

pure = pd.DataFrame(rows)
if len(pure):
    pure = pure.sort_values("p_by_chance").reset_index(drop=True)
    pure.to_csv(ROOT / "artifacts" / "pure_value_check.csv", index=False)
    print("pure values (n >= 5):", len(pure))
    print("\nfor minority classes only:")
    print(pure[pure["class"] != "Category_1"].head(20))
    print("\ntop by unlikeliness:")
    print(pure.head(15))
    print("\nrows covered by pure values, by column:")
    print(pure.groupby("col")["n"].sum())
else:
    print("no pure values above threshold")

pure values (n >= 5): 456

for minority classes only:
     col                                  value   n       class   p_by_chance
0   Col4                        Word633 Word575  56  Category_2  4.345642e-55
1   Col4                                Word575  50  Category_2  2.901003e-49
3   Col4        Word984 Word835 Word905 Word791   6  Category_3  3.513150e-15
5   Col4                        Word905 Word794   5  Category_3  9.010465e-13
7   Col1  Word199 Word200 Word201 Word202 Word7  12  Category_2  2.243942e-12
8   Col4                                Word868  12  Category_2  2.243942e-12
11  Col4                                Word644  10  Category_2  1.961145e-10
16  Col4                        Word575 Word868   8  Category_2  1.713988e-08
19  Col4        Word762 Word21 Word145 Word1051   6  Category_2  1.497979e-06
24  Col4                               Word1609   5  Category_2  1.400409e-05
25  Col4               Word1236 Word773 Word774   5  Category_2  1.400409e-05

top by un

Col6 first token as a hierarchy

In [17]:
first = df["Col6"].str.split().str[0]
print(first.value_counts().head(10))
print("\ndistinct first tokens:", first.nunique(), "of", df["Col6"].nunique(), "full values")
print("\nclass share by first token (n >= 30):")
keep = first.value_counts()[lambda s: s >= 30].index
print(pd.crosstab(first[first.isin(keep)], df["label"], normalize="index").mul(100).round(2))

Col6
Word563    4169
Word567     652
Word560     417
Word571     204
Word52      135
Word576     127
Word565      57
Word588      31
Word613      31
Word288      25
Name: count, dtype: int64

distinct first tokens: 24 of 196 full values

class share by first token (n >= 30):
label    Category_1  Category_2  Category_3  Category_4  Category_5  \
Col6                                                                  
Word52        91.85        7.41        0.00        0.00        0.74   
Word560       98.08        1.92        0.00        0.00        0.00   
Word563       86.16       13.26        0.43        0.07        0.00   
Word565       91.23        8.77        0.00        0.00        0.00   
Word567       97.09        2.91        0.00        0.00        0.00   
Word571       92.16        0.49        2.45        0.49        0.00   
Word576       78.74       12.60        0.00        8.66        0.00   
Word588       67.74       29.03        0.00        0.00        3.23   
Word613       

Col1 first and last tokens tested for the same structure

In [18]:
for pos, name in [(0, "first"), (-1, "last")]:
    tok = df["Col1"].str.split().str[pos]
    print(f"\n=== Col1 {name} token | distinct: {tok.nunique()}")
    print(tok.value_counts().head(8))
    keep = tok.value_counts()[lambda s: s >= 30].index
    print(pd.crosstab(tok[tok.isin(keep)], df["label"], normalize="index").mul(100).round(2))


=== Col1 first token | distinct: 277
Col1
Word16    1443
Word8      325
Word30     264
Word19     203
Word85     196
Word33     173
Word42     163
Word38     161
Name: count, dtype: int64
label    Category_1  Category_2  Category_4
Col1                                       
Word101       52.17       23.91       23.91
Word106      100.00        0.00        0.00
Word113      100.00        0.00        0.00
Word116      100.00        0.00        0.00
Word134      100.00        0.00        0.00
Word144       94.74        5.26        0.00
Word16        93.62        6.38        0.00
Word160      100.00        0.00        0.00
Word162      100.00        0.00        0.00
Word166      100.00        0.00        0.00
Word179      100.00        0.00        0.00
Word19         6.40       93.60        0.00
Word24       100.00        0.00        0.00
Word299      100.00        0.00        0.00
Word30       100.00        0.00        0.00
Word33       100.00        0.00        0.00
Word38       100.00